# L1 · Primer contacto — Sesión 2: **Responder**
### Introducción a las Ciencias de la Computación · UNAL Manizales

En la sesión 1 recibieron un mensaje. Hoy escriben uno: su especie le va a hablar a otra especie del salón, que lo va a leer **sin instrucciones**, mientras el canal le daña bits al azar.

Al final de la sesión viene **el día del contacto**: los mensajes se reparten y cada equipo intenta leer uno ajeno con las herramientas que construyó la semana pasada.

In [ ]:
EQUIPO = ["Emily Sofía Céspedes Quintero"]
MI_BASE = 6
NOMBRE_ESPECIE = "Cafeteros de la montaña"

## Momento 1 — Predecir *(en papel)*

✍️
**P4.** ¿Qué fracción de su mensaje creen que la otra especie va a entender? Escriban un número entre 0 y 1 y expliquen qué sección creen que se va a perder primero.

**Respuesta:** 0.75 (3 de las 4 secciones). La sección que creo que se va a perder primero es la cuarta (la taza de café humeante y los datos del curso), porque mientras el conteo en base 6, la silueta humana y las montañas tienen referentes físicos o matemáticos universales, una taza de café es un objeto cultural propio de nuestra especie y sin instrucciones es casi imposible que otra civilización deduzca qué significa.

**P5.** Si el canal daña 1 de cada 100 bits, ¿cuántos bits dañados esperan en su mensaje? ¿Y si lo mandan tres veces?

**Respuesta:** Como el mensaje tiene 437 bits (19 x 23) y el canal daña 1 de cada 100 bits (p = 0.01), esperamos 437 * 0.01 = 4.37 bits dañados en un envío (aproximadamente 4 o 5 bits). Si lo mandamos tres veces (1311 bits en total), esperamos 1311 * 0.01 = 13.11 bits dañados en el camino (unos 13 bits repartidos entre las tres copias), pero al comparar las tres copias por mayoría se espera recuperar el mensaje con 0 errores.

## 0. Sus herramientas

Peguen aquí **sus** funciones de la sesión 1 y de la tarea puente. Son suyas: se reutilizan, no se reescriben. Las pruebas de abajo verifican que llegaron completas.

In [ ]:
def es_primo(n):
    """CONTRATO
    Recibe: un entero n.
    Devuelve: True si n es primo, False si no.
    Supone: n >= 0.
    """
    es_primo = (n >= 2)
    d = 2
    while d * d <= n and es_primo:
        if n % d == 0:
            es_primo = False
        d = d + 1
    return es_primo


def rectangulos(n):
    """CONTRATO
    Recibe: un entero n > 1.
    Devuelve: las parejas (filas, columnas) cuyo producto es n,
    ordenadas de menor a mayor número de filas.
    """
    parejas = []
    for filas in range(1, n + 1):
        if n % filas == 0:
            columnas = n // filas
            parejas.append((filas, columnas))
    return parejas


def filas_de(bits, columnas):
    """CONTRATO
    Recibe: una cadena de bits y un número de columnas que divide su longitud.
    Devuelve: una lista de renglones, cada uno de longitud columnas.
    """
    renglones = []
    for i in range(0, len(bits), columnas):
        renglones.append(bits[i:i + columnas])
    return renglones


def dibujar(bits, columnas):
    """CONTRATO
    Recibe: una cadena de bits y un número de columnas.
    Devuelve: nada. Imprime 1 como bloque y 0 como espacio.
    """
    for fila in filas_de(bits, columnas):
        linea = fila.replace("1", "█").replace("0", " ")
        print(linea)


def leer_numero(lista_bits):
    """CONTRATO
    Recibe: una lista de enteros 0 y 1; el primero es el más significativo.
    Devuelve: el entero que representan en base 2.
    """
    valor = 0
    for bit in lista_bits:
        valor = valor * 2 + bit
    return valor


def columna(bits, columnas, j, fila_desde, fila_hasta):
    """CONTRATO
    Recibe: los bits, el ancho de la cuadrícula, una columna j y un rango de filas.
    Devuelve: los bits de esa columna desde fila_desde hasta fila_hasta - 1.
    """
    resultado = []
    for f in range(fila_desde, fila_hasta):
        indice = f * columnas + j
        resultado.append(int(bits[indice]))
    return resultado


def a_base_rec(n, b):
    """CONTRATO
    Recibe: un entero n >= 0 y una base b >= 2.
    Devuelve: la lista de dígitos de n en base b, del más significativo al menos significativo.
    """
    if n < b:
        return [n]
    return a_base_rec(n // b, b) + [n % b]


def de_base(digitos, b):
    """CONTRATO
    Recibe: una lista de dígitos en base b, el más significativo primero.
    Devuelve: el entero que representan.
    """
    numero = 0
    for d in digitos:
        numero = numero * b + d
    return numero

In [ ]:
# Pruebas de que las herramientas llegaron completas
assert es_primo(97) and not es_primo(1679)
assert rectangulos(12) == [(1, 12), (2, 6), (3, 4), (4, 3), (6, 2), (12, 1)]
assert filas_de("101100", 3) == ["101", "100"]
assert leer_numero([1, 0, 1, 1]) == 11
assert a_base_rec(1974, 6) == [1, 3, 0, 5, 0] and de_base([1, 3, 0, 5, 0], 6) == 1974
assert MI_BASE in (2, 3, 5, 6, 8, 12), "Escriban la base que les tocó."
print("✔ Herramientas listas. Especie de base", MI_BASE)

✔ Herramientas listas. Especie de base 6


## 1. ¿Cuántos bits por dígito?

Si cuentan en base 6, sus dígitos van de 0 a 5, y cada uno hay que escribirlo con bits. ¿Con cuántos? Es la desigualdad de la clase 3: con $k$ bits se escriben $2^k$ valores distintos, así que hace falta el menor $k$ con $2^k \geq b$.

- `bits_por_digito(b)` devuelve ese $k$.
- `digito_a_bits(d, ancho)` escribe un dígito con exactamente `ancho` bits, rellenando con ceros a la izquierda. Usen su `a_base_rec(d, 2)`.

In [ ]:
def bits_por_digito(b):
    """CONTRATO. Recibe una base b >= 2. Devuelve el menor k con 2**k >= b."""
    k = 1
    while 2 ** k < b:
        k += 1
    return k


def digito_a_bits(d, ancho):
    """CONTRATO. Recibe un entero 0 <= d < 2**ancho. Devuelve una cadena de exactamente
    'ancho' caracteres '0'/'1' con d en binario, rellenada con ceros a la izquierda."""
    bits = "".join(str(x) for x in a_base_rec(d, 2))
    return "0" * (ancho - len(bits)) + bits

In [ ]:
# Pruebas
assert [bits_por_digito(b) for b in (2, 3, 5, 6, 8, 12)] == [1, 2, 3, 3, 3, 4]
assert digito_a_bits(5, 3) == "101" and digito_a_bits(1, 4) == "0001" and digito_a_bits(0, 2) == "00"
ANCHO = bits_por_digito(MI_BASE)
print("✔ Su especie escribe cada dígito con", ANCHO, "bits.")

✔ Su especie escribe cada dígito con 3 bits.


## 2. El mensaje, por secciones

**La representación del mensaje.** Un mensaje es una **lista de renglones**, y cada renglón es una cadena de `'0'` y `'1'` de exactamente `COLUMNAS` caracteres. Elijan `COLUMNAS` pensando en la pregunta de Arecibo: ¿cómo va a adivinar la otra especie el ancho del rectángulo?

Cada sección del mensaje es una **función** que devuelve una lista de renglones. Al final, `ensamblar` las pone una debajo de la otra. Si mañana quieren cambiar una sección, cambian una función y nada más: eso es pensar en módulos.

### La sección obligatoria: contar

`codificar_conteo(hasta)` escribe los números del 1 al `hasta` **en su base**, uno por renglón, cada dígito con `ANCHO` bits, alineados a la izquierda y rellenando el renglón con ceros. Ojo: deben contar al menos hasta un poco más allá de su base, para que se vea el momento en que aparece el "10".

In [ ]:
COLUMNAS = 19  # elijan el ancho del mensaje


def renglon(contenido):
    """CONTRATO. Recibe una cadena de bits de longitud <= COLUMNAS.
    Devuelve esa cadena rellenada con ceros a la derecha hasta COLUMNAS caracteres."""
    assert len(contenido) <= COLUMNAS, f"El renglón no cabe: {len(contenido)} > {COLUMNAS}"
    return contenido + "0" * (COLUMNAS - len(contenido))


def codificar_conteo(hasta):
    """CONTRATO. Devuelve una lista de renglones: el renglón i tiene el número i
    (de 1 a 'hasta') escrito en MI_BASE, cada dígito con ANCHO bits."""
    renglones = []
    for i in range(1, hasta + 1):
        digitos = a_base_rec(i, MI_BASE)
        bits = "".join(digito_a_bits(d, ANCHO) for d in digitos)
        renglones.append(renglon(bits))
    return renglones

In [ ]:
# Pruebas
conteo = codificar_conteo(MI_BASE + 2)
assert len(conteo) == MI_BASE + 2 and all(len(r) == COLUMNAS for r in conteo)
assert conteo[0].startswith(digito_a_bits(1, ANCHO))
assert conteo[MI_BASE - 1].startswith(digito_a_bits(1, ANCHO) + digito_a_bits(0, ANCHO)), "El renglón de MI_BASE debe decir '10'."
print("✔ El conteo delata su base. Así se ve:")
dibujar("".join(conteo), COLUMNAS)

✔ El conteo delata su base. Así se ve:
  █                
 █                 
 ██                
█                  
█ █                
  █                
  █  █             
  █ █              


### Sus secciones

Una función por sección, cada una con su contrato. Copien la plantilla tantas veces como secciones tengan.

In [ ]:
def numero_a_bits(n):
    """Convierte el entero n a base MI_BASE y cada dígito a ANCHO bits."""
    return "".join(digito_a_bits(d, ANCHO) for d in a_base_rec(n, MI_BASE))


def codificar_seccion_2():
    """CONTRATO. Qué dice: Silueta humana y anatomía (2 extremidades, 5 dedos/mano, 20 dedos totales).
    Devuelve: lista de 5 renglones de COLUMNAS bits."""
    silueta = [
        "0000000001000000000",  #         █
        "0000000111110000000",  #       █████
        "0000000001000000000",  #         █
        "0000000010100000000",  #        █ █
    ]
    datos_anatomia = renglon(numero_a_bits(2) + numero_a_bits(5) + numero_a_bits(20))
    seccion = silueta + [datos_anatomia]
    assert all(len(r) == COLUMNAS for r in seccion)
    return seccion


def codificar_seccion_3():
    """CONTRATO. Qué dice: Montañas y altitudes de Manizales (2150 m) y el Nevado (5321 m) en base 6.
    Devuelve: lista de 5 renglones de COLUMNAS bits."""
    montanas = [
        "0000000010000000000",  #        █
        "0000000111000100000",  #       ███   █
        "0000001111101110000",  #      █████ ███
    ]
    alt_manizales = renglon(numero_a_bits(2150))
    alt_nevado = renglon(numero_a_bits(5321))
    seccion = montanas + [alt_manizales, alt_nevado]
    assert all(len(r) == COLUMNAS for r in seccion)
    return seccion


def codificar_seccion_4():
    """CONTRATO. Qué dice: Taza de café humeante (cultura cafetera), estudiantes del curso (25) y equipo (1).
    Devuelve: lista de 5 renglones de COLUMNAS bits."""
    cafe = [
        "0000000101010000000",  #       █ █ █
        "0000011111111110000",  #     ██████████
        "0000011111110010000",  #     ███████  █
        "0000001111100000000",  #      █████
    ]
    datos_curso = renglon(numero_a_bits(25) + numero_a_bits(1))
    seccion = cafe + [datos_curso]
    assert all(len(r) == COLUMNAS for r in seccion)
    return seccion


In [ ]:
def ensamblar(secciones):
    """CONTRATO. Recibe una lista de secciones (cada una, una lista de renglones).
    Devuelve una sola cadena de bits: todos los renglones, uno detrás de otro."""
    return "".join("".join(sec) for sec in secciones)


mensaje = ensamblar([
    codificar_conteo(MI_BASE + 2),
    codificar_seccion_2(),
    codificar_seccion_3(),
    codificar_seccion_4(),
])
print("Longitud del mensaje:", len(mensaje), "bits")
print("Formas de dibujarlo:", rectangulos(len(mensaje)))
dibujar(mensaje, COLUMNAS)

Longitud del mensaje: 437 bits
Formas de dibujarlo: [(1, 437), (19, 23), (23, 19), (437, 1)]
  █                
 █                 
 ██                
█                  
█ █                
  █                
  █  █             
  █ █              
         █         
       █████       
         █         
        █ █        
 █ █ █ ██ █        
        █          
       ███   █     
      █████ ███    
  █ ███ ██   █     
█      ███  █ █    
       █ █ █       
     ██████████    
     ███████  █    
      █████        
█    █  █          


### La forma

¿Su mensaje tiene una longitud que obligue a **una sola** manera de dibujarlo, como Arecibo? Si no, agreguen renglones de relleno hasta que el número de renglones lo permita. Pista: si `COLUMNAS` es primo y el número de renglones también, solo hay dos rectángulos posibles (y uno es el traspuesto del otro). Escriban `es_semiprimo(n)` con su `rectangulos` si les sirve.

In [ ]:
def es_semiprimo(n):
    """CONTRATO. Devuelve True si n es el producto de dos números primos."""
    for filas, cols in rectangulos(n):
        if es_primo(filas) and es_primo(cols):
            return True
    return False


print("Longitud del mensaje:", len(mensaje))
print("¿Es semiprimo?:", es_semiprimo(len(mensaje)))

Longitud del mensaje: 437
¿Es semiprimo?: True


## 3. Las preguntas incómodas de la representación

Su mensaje tiene que responder **al menos dos** de estas. En cada una, escriban cómo lo resolvieron (o por qué no se puede).

1. **Ancho fijo.** Si cada número va en un renglón de `COLUMNAS` bits, ¿cuál es el número más grande que cabe en su base? ¿Qué pasa si quieren decir cuántos habitantes tiene Manizales? A esto se le llama **desbordamiento**.
2. **Decimales.** ¿Cómo dicen la altura de Manizales en metros si el otro no sabe qué es un metro, ni dónde va la coma? *(Pista real: en 1972, las placas de las sondas Pioneer usaron una propiedad del átomo de hidrógeno como regla para medir, porque el hidrógeno es igual en todo el universo.)*
3. **Texto.** ¿Por qué no pueden mandar la palabra "Manizales" en ASCII o en Unicode? ¿Qué **sí** pueden mandar?
4. **Forma.** ¿Cómo va a saber la otra especie cuántas columnas tiene su mensaje?

In [ ]:
digitos_por_renglon = COLUMNAS // ANCHO
maximo = MI_BASE ** digitos_por_renglon - 1
poblacion_manizales = 450000
digitos_poblacion = len(a_base_rec(poblacion_manizales, MI_BASE))

print("Dígitos en base 6 que caben por renglón:", digitos_por_renglon)
print("Número más grande que cabe en un renglón:", maximo)
print("Dígitos que necesita la población de Manizales:", digitos_poblacion, f"({digitos_poblacion * ANCHO} bits)")

Dígitos en base 6 que caben por renglón: 6
Número más grande que cabe en un renglón: 46655
Dígitos que necesita la población de Manizales: 8 (24 bits)


✍️
1. Ancho fijo: Como cada renglón mide 19 bits y cada dígito en base 6 ocupa 3 bits, en un solo renglón caben máximo $19 // 3 = 6$ dígitos. El número más grande que podemos escribir ahí es $6^6 - 1 = 46\,655$. Si quisiéramos mandar los habitantes de Manizales (unos $450\,000$), ocurriría un desbordamiento, porque en base 6 ese número es 13351200 ($8$ dígitos = $24$ bits) y no cabe en una sola fila. Por eso en nuestro mensaje usamos números más pequeños que sí cupieran, como las altitudes ($2150$ y $5321$) y la cantidad de estudiantes ($25$).   
2. Decimales: No podemos mandar la coma decimal ni asumir que la otra especie conoce el metro, porque el metro es una medida humana arbitraria. Como solución, usamos únicamente números enteros y dibujamos una referencia física visual (la montaña junto a la silueta) para dar contexto de escala; si quisiéramos dar medidas exactas en el espacio, tendríamos que expresarlas como múltiplos enteros de una constante física universal, como la longitud de onda del hidrógeno que usaron las sondas Pioneer
3. Texto: No podemos mandar "Manizales" en ASCII o Unicode porque esas tablas son acuerdos exclusivos de los computadores humanos: un receptor extraterrestre no conoce nuestro alfabeto, nuestro idioma ni qué número le asignamos a cada letra. Lo que sí podemos mandar son números naturales (mediante conteo), relaciones matemáticas y dibujos en cuadrícula de cosas físicas (nuestro cuerpo, las montañas, la taza de café).   
4. Forma: La otra especie sabrá que el mensaje tiene 19 columnas porque la longitud total es 437 bits, que es un número semiprimo ($19 \times 23$). Al buscar sus divisores, solo existen dos cuadrículas posibles ($19 \times 23$ o $23 \times 19$, sin contar una sola línea): al probar ambas, una se ve como puro ruido desordenado y la de 19 columnas revela de inmediato el conteo ordenado y los dibujos.

## 4. El canal

Entre las estrellas el mensaje se daña: la radiación voltea bits. Vamos a simular ese canal y a comparar **dos maneras** de proteger el mensaje.

- `ruido(bits, p)`: voltea cada bit con probabilidad `p` (usen `random.random() < p`).
- **Manera 1, paridad.** `con_paridad(bits, ancho)` agrega a cada renglón un bit extra para que el número de unos del renglón sea **par**. Del otro lado, `filas_danadas(bits, ancho)` dice qué renglones llegaron con número impar de unos. (Ojo: el ancho de cada renglón ahora es `ancho + 1`.)
- **Manera 2, repetición.** `repetir3(bits)` manda cada bit tres veces seguidas; `votar(bits)` toma cada grupo de tres y se queda con el que más aparece.

In [ ]:
import random


def ruido(bits, p):
    """CONTRATO. Recibe una cadena de bits y una probabilidad p (0 <= p <= 1).
    Devuelve una cadena donde cada bit se invierte con probabilidad p."""
    resultado = []
    for b in bits:
        if random.random() < p:
            resultado.append("1" if b == "0" else "0")
        else:
            resultado.append(b)
    return "".join(resultado)


def con_paridad(bits, columnas):
    """CONTRATO. Agrega 1 bit de paridad par al final de cada fila de ancho 'columnas'.
    Devuelve una cadena con filas de ancho 'columnas + 1'."""
    filas_nuevas = []
    for fila in filas_de(bits, columnas):
        bit_p = "1" if fila.count("1") % 2 == 1 else "0"
        filas_nuevas.append(fila + bit_p)
    return "".join(filas_nuevas)


def filas_danadas(bits_paridad, columnas):
    """CONTRATO. Recibe un mensaje con paridad (filas de ancho 'columnas + 1').
    Devuelve la lista de índices de las filas cuya paridad quedó impar."""
    danadas = []
    for i, fila in enumerate(filas_de(bits_paridad, columnas + 1)):
        if fila.count("1") % 2 == 1:
            danadas.append(i)
    return danadas


def repetir3(bits):
    """CONTRATO. Repite cada bit de la cadena 3 veces seguidas."""
    return "".join(b * 3 for b in bits)


def votar(bits3):
    """CONTRATO. Recibe una cadena con bits triplicados y recupera cada bit
    eligiendo el valor mayoritario en cada bloque de 3 bits."""
    recuperado = []
    for i in range(0, len(bits3), 3):
        bloque = bits3[i:i + 3]
        recuperado.append("1" if bloque.count("1") >= 2 else "0")
    return "".join(recuperado)

In [ ]:
# Pruebas
random.seed(0)
assert ruido("0101", 0) == "0101" and ruido("0101", 1) == "1010"
assert con_paridad("110100", 3) == "11001001"
assert filas_danadas("11001001", 3) == [] and filas_danadas("11011001", 3) == [0]
assert repetir3("10") == "111000" and votar("110001") == "10" and votar(repetir3("1011")) == "1011"
print("✔ El canal y las dos protecciones pasan las pruebas.")

✔ El canal y las dos protecciones pasan las pruebas.


### Medir, no suponer

Corran el canal **mil veces** con `p = 0.01` sobre su mensaje y cuenten:

- **Paridad:** ¿en cuántas transmisiones hubo al menos un error? ¿En cuántas `filas_danadas` lo detectó? ¿En cuántas hubo errores que **no** detectó? (¿Cómo puede pasar eso?)
- **Repetición:** ¿en cuántas transmisiones `votar` recuperó el mensaje **exacto**?
- ¿Cuántos bits cuesta cada manera, comparado con el mensaje original?

In [ ]:
N = 1000
p = 0.01

con_error = detectadas = no_detectadas = 0
recuperadas_rep = 0

for _ in range(N):
    protegido = con_paridad(mensaje, COLUMNAS)
    recibido = ruido(protegido, p)
    if recibido != protegido:
        con_error += 1
        if len(filas_danadas(recibido, COLUMNAS)) > 0:
            detectadas += 1
        else:
            no_detectadas += 1

    protegido_rep = repetir3(mensaje)
    recibido_rep = ruido(protegido_rep, p)
    if votar(recibido_rep) == mensaje:
        recuperadas_rep += 1

print("Paridad   : con error", con_error, "| detectadas", detectadas, "| sin detectar", no_detectadas)
print("Repetición: recuperadas exactas", recuperadas_rep, "de", N)
print("Costo en bits: original", len(mensaje), "| paridad", len(con_paridad(mensaje, COLUMNAS)),
      "| repetición", len(repetir3(mensaje)))

Paridad   : con error 983 | detectadas 979 | sin detectar 4
Repetición: recuperadas exactas 887 de 1000
Costo en bits: original 437 | paridad 460 | repetición 1311


✍️ **¿Cuál de las dos protecciones elegirían para un mensaje que tarda 25 000 años en llegar, y por qué?** Con sus números. Piensen en la diferencia entre **detectar** un error y **corregirlo**.

**Respuesta:** Elegiría la protección por repetición (repetir3 + votar).
La diferencia clave está entre detectar un error y corregirlo: con paridad el costo es muy bajo (460 bits frente a los 437 originales) y detecta casi todos los fallos (979 de 983 envíos con error, con solo 4 sin detectar), pero no permite reparar los bits dañados. En un mensaje que tarda 25 000 años en llegar, el receptor no puede pedir retransmisión (esperar una respuesta tomaría 50 000 años).

En cambio, aunque la repetición triplica el costo a 1311 bits, le da al receptor la capacidad de corregir los errores por mayoría en el destino, recuperando el mensaje completamente intacto en 887 de 1000 envíos (88.7 %), frente a apenas 17 envíos intactos (1.7 %) si no se corrige nada. Cuando solo hay una oportunidad de comunicarse a esa distancia, el costo en bits se justifica por completo.

## 5. Mandar el mensaje

Elijan su protección, pasen el mensaje por el canal **una sola vez** con `p = 0.01` y guárdenlo en `mensaje.txt`: una sola línea de ceros y unos, **sin** decir cuántas columnas tiene ni qué protección usaron. Entréguenle el archivo al monitor. Súbanlo también a su repositorio.

In [ ]:
PROTECCION = "repeticion"

if PROTECCION == "paridad":
    enviado = ruido(con_paridad(mensaje, COLUMNAS), 0.01)
else:
    enviado = ruido(repetir3(mensaje), 0.01)

with open("mensaje.txt", "w") as f:
    f.write(enviado)
print("mensaje.txt listo:", len(enviado), "bits. Entréguenlo al monitor.")

mensaje.txt listo: 1311 bits. Entréguenlo al monitor.


## 6. El día del contacto

El monitor les entrega el `mensaje.txt` de **otra especie**. Tienen 30 minutos para leerlo **con sus herramientas de la sesión 1**: `len`, `rectangulos`, `dibujar`, `columna`, `leer_numero`, `de_base`. Si sirven para un mensaje que nunca habían visto, eran herramientas de verdad; si solo servían para Arecibo, eran un truco.

Súbanlo a Colab (ícono de carpeta → subir) con el nombre `recibido.txt`.

In [ ]:
with open("recibido.txt") as f:
    ajeno = f.read().strip()

print("Longitud:", len(ajeno))
print("Formas posibles:", rectangulos(len(ajeno)))

limpio = votar(ajeno)
print("Longitud limpia:", len(limpio))
print("Formas del mensaje limpio:", rectangulos(len(limpio)))

COLUMNAS_AJENO = 47
dibujar(limpio, COLUMNAS_AJENO)

print("\nLectura del conteo (filas 0 a 9, bloques de 3 bits):")
for fila in range(10):
    d1 = leer_numero([columna(limpio, COLUMNAS_AJENO, j, fila, fila + 1)[0] for j in range(0, 3)])
    d2 = leer_numero([columna(limpio, COLUMNAS_AJENO, j, fila, fila + 1)[0] for j in range(3, 6)])
    if d2 == 0 and fila < 7:
        print(f"Fila {fila}: [{d1}]")
    else:
        print(f"Fila {fila}: [{d1}, {d2}] -> en base 8 equivale a {de_base([d1, d2], 8)}")

Longitud: 7473
Formas posibles: [(1, 7473), (3, 2491), (47, 159), (53, 141), (141, 53), (159, 47), (2491, 3), (7473, 1)]
Longitud limpia: 2491
Formas del mensaje limpio: [(1, 2491), (47, 53), (53, 47), (2491, 1)]
  █                                            
 █                                             
 ██                                            
█                                              
█ █                                            
██                                             
███                                            
  █                                            
  █  █                                         
  █ █                                          
                                               
█████████████████████                          
█    █  █   █ ███   █                          
█ ██ █ ██ █ █ █     █                          
█ █████ █ █ █ █ █   █                          
█ █   ███ █ █   █████                          
█ ████  █ █ █ █ █  

✍️ **Lo que entendimos del mensaje de la especie Glitx Synkai :**

- **Ancho que se dedujo:** `47` columnas, porque al quitar la protección de repetición por 3 (`7473 / 3 = 2491` bits), `2491` es un semiprimo que solo se puede acomodar en `47 x 53` o `53 x 47`. Al dibujarlo con `47` columnas todas las figuras y números quedan bien alineados, mientras que con `53` se ve todo cortado en diagonal.
- **Base que se dedujo:** `8`, porque en las primeras 10 filas aparece el conteo escrito de a 3 bits por dígito: del `1` al `7` (`001` hasta `111`) ocupan un solo dígito, y justo en la octava fila aparece `[1, 0]` (`001 000`), lo que demuestra que su base es 8 (y siguen con `11` y `12`, que son 9 y 10).
- **Protección que creo que usaron:** Repetición por 3 (`repetir3`), porque la longitud del archivo (`7473`) es exactamente el triple de `2491` y los bits vienen agrupados en tríos iguales (con algunos errores aislados que `votar()` corrige sin problema).
- **Lo que dice cada sección:**
  1. **Filas 0 a 9 (Conteo):** Los números del 1 al 10 escritos en base 8 (`1, 2, 3, 4, 5, 6, 7, 10, 11, 12`), usando 3 bits por cada dígito de izquierda a derecha.
  2. **Filas 11 a 31 (Código QR):** Un código QR de `21 x 21` píxeles con sus tres cuadros guía en las esquinas, pensado para escanearlo y acceder a información externa.
  3. **Filas 33 a 41 (Mirar / dirección):** Un rectángulo o pantalla a la izquierda, un par de ojos mirando hacia la derecha y una flecha apuntando hacia la derecha (`→`).
  4. **Filas 44 a 52 (Observador):** Otro par de ojos mirando hacia la izquierda, un cuadro en el centro y la silueta de una persona a la derecha representando a su especie.

## 7. La revelación

Cada especie lee en voz alta lo que entendió del mensaje que recibió, y la especie emisora revela su contrato de lectura. Anoten **de su propio mensaje**:

| Paso de nuestro contrato de lectura | ¿Lo dio la otra especie? | Si no, ¿por qué? |
|---|---|---|
| | | |
| | | |
| | | |

¿Hubo algo que la otra especie entendió **mal**, con toda seguridad? (Recuerden la palabra "arma" en *La llegada*.)

## Momento 3 — Explicar *(una página por equipo)*

1. **Confronten P4** con lo que de verdad se entendió de su mensaje. ¿Por qué la diferencia?
2. **La convención escondida:** ¿qué supusieron **sin darse cuenta** que la otra especie no tenía?
3. **Paridad contra repetición:** cuál eligieron y por qué, con las mediciones de la sección 4.
4. Una frase que empiece por *"Representar es..."*. Compárenla con la de la sesión 1: ¿cambió?

✍️

## La IA como objeto de estudio *(para la bitácora individual)*

Denle a su asistente el `mensaje.txt` de **otra especie** del salón (no el suyo), sin decirle nada más, y pídanle que lo traduzca. Ese mensaje no está en ningún lugar de internet: no lo puede reconocer, solo lo puede descifrar.

Comparen lo que dice el asistente con lo que reveló la especie emisora. ¿Tradujo con seguridad algo que no estaba ahí? Esa es la escena del "arma", y la acaban de medir. Y vuelvan a la pregunta de la sesión 1: ¿qué hizo con Arecibo, descifrar o reconocer?

## Foto de las predicciones

`foto_predicciones_sesion2.jpg`

---
### Antes de irse
1. **Ejecuten todo de arriba a abajo** (*Entorno de ejecución → Reiniciar y ejecutar todo*). Si algo falla, no está terminado.
2. **Guarden una copia en su repositorio `intro-cc`:** *Archivo → Guardar una copia en GitHub*. Cada integrante en el suyo.
3. La **bitácora de IA** es individual y va aparte (la plantilla es `bitacora_IA_L1.md`, en la carpeta `introduccion_cc` del repositorio de laboratorios).